# Car Specs — Consumer Decision EDA

Structured around the 5 factors that drive car purchase decisions:
1. **Budget** — price & fuel efficiency
2. **Performance** — horsepower, 0-60, torque
3. **Vehicle Type & Practicality** — bodytype, drivetrain, cargo
4. **Safety & Features** — feature availability, luxury gap
5. **Brand & Quality** — EV adoption, brand reputation

In [3]:
import pandas as pd
from pathlib import Path
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
BASE = Path('/')

In [4]:
cars     = pd.read_csv(BASE / 'data/final/cars/car_stats.UVCRS')
features = pd.read_csv(BASE / 'data/final/cars/car_features.UVCRS')
# z60      = pd.read_csv(BASE / 'data/final/cars/zero_to_sixty_est.UVCRS')[['car_id', 'est_0_60', 'SUSPECT_HP_OR_WEIGHT']]
reviews  = pd.read_csv(BASE / 'data/final/reviews/review_ratings.UVCRS')

print(f'cars     : {cars.shape}')
print(f'features : {features.shape}')
# print(f'z60      : {z60.shape}')
print(f'reviews  : {reviews.shape}')

cars     : (27902, 30)
features : (27902, 43)
reviews  : (4782, 7)


In [9]:
cars['make'].unique()

array(['Chrysler', 'Dodge', 'Ford', 'Honda', 'Mazda', 'Nissan',
       'Plymouth', 'Toyota', 'Volkswagen', 'Acura', 'Audi', 'BMW',
       'Chevrolet', 'GMC', 'Hyundai', 'Isuzu', 'Jaguar', 'Jeep', 'Kia',
       'Lexus', 'Lincoln', 'Mercedes-Benz', 'Mercury', 'Mitsubishi',
       'Porsche', 'Subaru', 'Volvo', 'Cadillac', 'Land-Rover', 'Mini',
       'London Taxi', 'Buick', 'Maserati', 'Saturn', 'Pontiac', 'Hummer',
       'Bentley', 'Mahindra', 'Saab', 'smart', 'Azure Dynamics', 'BYD',
       'CODA Automotive', 'Fiat', 'Fisker', 'Infiniti', 'Ram', 'Tesla',
       'Scion', 'Ferrari', 'McLaren Automotive', 'Alfa-Romeo', 'Genesis',
       'Karma', 'Koenigsegg', 'Polestar', 'Kandi', 'Lucid', 'Rivian',
       'Lordstown', 'Vinfast', 'Bugatti Rimac', 'Lamborghini',
       'Rolls-Royce'], dtype=object)

In [142]:
# Consumer-relevant columns — focus of this analysis
CONSUMER_COLS = [
    'car_id', 'make', 'model', 'year', 'trim', 'bodytype', 'doors',
    'price', 'Fuel Type', 'mpg_city', 'mpg_hwy', 'mpg_comb',
    'hp', 'torque_lbft', '0 - 60', 'Drivetrain', 'Transmission Type',
    'cargo_cuft',
]

NON_CONSUMER = [
    'Engine', 'Recommended Fuel', 'curb_weight_lbs', 'Fuel Capacity',
    'Wheel Base', 'Overall Length', 'Front Head Room', 'Front Leg Room',
    'Front Shoulder Room', 'Width with mirrors', 'Turning Diameter',
]

print(f'Total columns in raw data : {len(cars.columns)}')
print(f'Consumer-relevant kept    : {len(CONSUMER_COLS)}')
print(f'Technical specs dropped   : {len(NON_CONSUMER)}')
print(f'\nDropped: {NON_CONSUMER}')

Total columns in raw data : 30
Consumer-relevant kept    : 18
Technical specs dropped   : 11

Dropped: ['Engine', 'Recommended Fuel', 'curb_weight_lbs', 'Fuel Capacity', 'Wheel Base', 'Overall Length', 'Front Head Room', 'Front Leg Room', 'Front Shoulder Room', 'Width with mirrors', 'Turning Diameter']


In [143]:
# Null counts for consumer columns (before any imputation)
PREVIEW_COLS = [
    'price', 'mpg_comb', 'hp', '0 - 60', 'torque_lbft',
    'Fuel Type', 'Drivetrain', 'Transmission Type', 'bodytype', 'doors', 'cargo_cuft',
]

print(f'Raw shape: {cars.shape}')
null_before = cars[PREVIEW_COLS].isnull().sum().sort_values(ascending=False)
pct_before  = (null_before / len(cars) * 100).round(1)
print('\nNull counts (consumer cols, before imputation):')
pd.concat([null_before.rename('null_count'), pct_before.rename('null_%')], axis=1)

Raw shape: (27902, 30)

Null counts (consumer cols, before imputation):


,null_count,null_%
0 - 60,20749,74.4
cargo_cuft,9041,32.4
torque_lbft,3490,12.5
Transmission Type,3351,12.0
hp,3334,11.9
price,3091,11.1
doors,3030,10.9
bodytype,2914,10.4
mpg_comb,2111,7.6
Drivetrain,4,0.0


## Flags

In [144]:
cars['is_ev'] = cars['Fuel Type'].str.contains('Electric', case=False, na=False)

In [145]:
luxury = [
    'Acura', 'Alfa-Romeo', 'Audi', 'Bentley', 'BMW', 'Cadillac', 'Ferrari',
    'Genesis', 'Infiniti', 'Jaguar', 'Lamborghini', 'Land-Rover', 'Lexus',
    'Lincoln', 'Lucid', 'Maserati', 'McLaren Automotive', 'Mercedes-Benz',
    'Polestar', 'Porsche', 'Rivian', 'Rolls-Royce', 'Tesla', 'Volvo'
]
cars['luxury'] = cars['make'].isin(luxury).astype(int)

---

## Data Preparation

Impute **consumer-relevant columns only**, then filter rows.
Non-consumer technical specs (`Engine`, dimensions, `Recommended Fuel`, etc.) are left as-is.

### Doors

In [146]:
doors_missing = (
    cars
    .groupby('bodytype')['doors']
    .apply(lambda x: x.isna().sum())
)
print(doors_missing[doors_missing > 0])

bodytype
Hatchback     2
Minivan       8
Pickup       95
SUV          11
Name: doors, dtype: int64


In [147]:
four_doors = 'Double Cab|Quad Cab|Super Cab|Crew Cab|Flex Titanium|Extended Cab|Suburban|Yukon XL|Sienna|Sequoia|Sedona|RLX|MDX|ZDX|'
two_doors  = 'King Cab|XtraCab|Access Cab|Regular Cab|Club Cab|Clubman John Cooper'
three_doors = 'Accent Blue'

cars.loc[
    (cars['trim'].str.contains(four_doors, case=False, na=False)) & (cars['doors'].isna()),
    'doors'
] = 4
cars.loc[
    (cars['trim'].str.contains(two_doors, case=False, na=False)) & (cars['doors'].isna()),
    'doors'
] = 2
cars.loc[
    (cars['trim'].str.contains(three_doors, case=False, na=False)) & (cars['doors'].isna()),
    'doors'
] = 3

print(f'Remaining door nulls: {cars["doors"].isna().sum()}')

Remaining door nulls: 0


### Price

In [148]:
cars['price'] = cars.groupby(['make', 'bodytype', 'year'])['price'].transform(
    lambda x: x.fillna(x.median())
)
print(f'Remaining price nulls: {cars["price"].isna().sum()}')

Remaining price nulls: 2914


### mpg_comb

Fill where it can be derived from city + highway. `hp` and `torque_lbft` nulls are **flagged, not imputed** — they are model-specific and imputing would mislead consumers.

In [149]:
fill_mask = cars['mpg_comb'].isna() & cars['mpg_city'].notna() & cars['mpg_hwy'].notna()
cars.loc[fill_mask, 'mpg_comb'] = (
    (cars.loc[fill_mask, 'mpg_city'] + cars.loc[fill_mask, 'mpg_hwy']) / 2
).round(1)

# Null flags for hp and torque (not imputed)
cars['hp_missing']      = cars['hp'].isna().astype(int)
cars['torque_missing']  = cars['torque_lbft'].isna().astype(int)

print(f'mpg_comb filled from city+hwy : {fill_mask.sum():,}')
print(f'mpg_comb nulls remaining      : {cars["mpg_comb"].isna().sum():,}')
print(f'hp nulls (flagged)            : {cars["hp_missing"].sum():,}')
print(f'torque nulls (flagged)        : {cars["torque_missing"].sum():,}')

mpg_comb filled from city+hwy : 0
mpg_comb nulls remaining      : 2,111
hp nulls (flagged)            : 3,334
torque nulls (flagged)        : 3,490


### Drivetrain

In [150]:
cars['Drivetrain'] = cars.groupby(['make', 'bodytype'], dropna=False)['Drivetrain'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)
print(f'Remaining Drivetrain nulls: {cars["Drivetrain"].isna().sum()}')

Remaining Drivetrain nulls: 0


### Transmission Type

In [151]:
cars['Transmission Type'] = cars.groupby(['make', 'bodytype'], dropna=False)['Transmission Type'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)
cars['Transmission Type'] = cars.groupby('make')['Transmission Type'].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
)
print(f'Remaining Transmission Type nulls: {cars["Transmission Type"].isna().sum()}')

Remaining Transmission Type nulls: 502


/var/folders/9s/g28llhj16vvg6x74d1s9gr500000gn/T/ipykernel_4945/2863702193.py:2: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)
/var/folders/9s/g28llhj16vvg6x74d1s9gr500000gn/T/ipykernel_4945/2863702193.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  lambda x: x.fillna(x.mode()[0] if not x.mode().empty else x)


### Row Filter

Drop rows where `mpg_comb` is still null after the fill step, **excluding** EVs and hydrogen fuel cell vehicles (no traditional MPG, but valid consumer options).

In [152]:
EV_TYPES = {'Electric', 'Hydrogen Fuel Cell'}
drop_mask = cars['mpg_comb'].isna() & ~cars['Fuel Type'].isin(EV_TYPES)

before = len(cars)
cars = cars[~drop_mask].reset_index(drop=True)

print(f'Dropped : {drop_mask.sum():,} rows ({drop_mask.sum()/before:.1%}) — no MPG and not EV/H2FC')
print(f'Remaining: {len(cars):,} rows')

Dropped : 1,657 rows (5.9%) — no MPG and not EV/H2FC
Remaining: 26,245 rows


### Price Fallback

Two additional tiers for rows still null after the initial `(make, bodytype, year)` pass.

In [153]:
cars['price'] = cars.groupby(['make', 'bodytype'], dropna=False)['price'].transform(
    lambda x: x.fillna(x.median())
)
cars['price'] = cars.groupby('make')['price'].transform(
    lambda x: x.fillna(x.median())
)
print(f'Remaining price nulls: {cars["price"].isna().sum()}')

Remaining price nulls: 170


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  return np.nanmean(a, axis, out=out, keepdims=keepdims)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/numpy/lib/_nanfunctions_impl.py:1241: RuntimeWarning: Mean of empty slice
  

### Consumer Columns — Null Summary (after preparation)

In [154]:
CONSUMER_CHECK = [
    'price', 'mpg_comb', 'hp', 'torque_lbft',
    'Fuel Type', 'Drivetrain', 'Transmission Type', 'bodytype', 'doors', 'cargo_cuft',
]

null_after = cars[CONSUMER_CHECK].isnull().sum()
pct_after  = (null_after / len(cars) * 100).round(1)

print(f'Total rows: {len(cars):,}')
result = pd.concat([null_after.rename('null_count'), pct_after.rename('null_%')], axis=1)
result[result['null_count'] > 0].sort_values('null_%', ascending=False)

Total rows: 26,245


,null_count,null_%
cargo_cuft,7983,30.4
torque_lbft,3407,13.0
hp,3257,12.4
bodytype,2914,11.1
Transmission Type,502,1.9
mpg_comb,454,1.7
price,170,0.6


In [155]:
print('=== PRICE ===')
print(f'Range  : ${cars["price"].min():,.0f} – ${cars["price"].max():,.0f}')
print(f'Median : ${cars["price"].median():,.0f}')
print()
print('Median price by bodytype:')
print(cars.groupby('bodytype')['price'].median().sort_values(ascending=False).apply(lambda x: f'${x:,.0f}'))
print()
print('Luxury vs Non-Luxury:')
print(cars.groupby('luxury')['price'].median().rename({0: 'Non-Luxury', 1: 'Luxury'}).apply(lambda x: f'${x:,.0f}'))

=== PRICE ===
Range  : $1,125 – $687,000
Median : $14,750

Median price by bodytype:
bodytype
Coupe          $23,400
Convertible    $20,600
SUV            $17,550
Pickup         $15,250
Van            $14,000
Sedan          $12,100
Minivan        $10,100
Hatchback       $9,712
Wagon           $8,150
Name: price, dtype: object

Luxury vs Non-Luxury:
luxury
Non-Luxury    $12,450
Luxury        $20,900
Name: price, dtype: object


In [156]:
print('=== FUEL EFFICIENCY (mpg_comb) ===')
print(f'Range  : {cars["mpg_comb"].min():.0f} – {cars["mpg_comb"].max():.0f} mpg')
print(f'Median : {cars["mpg_comb"].median():.0f} mpg')
print()
print('Median MPG by Fuel Type:')
print(cars.groupby('Fuel Type')['mpg_comb'].median().sort_values(ascending=False).round(1))

=== FUEL EFFICIENCY (mpg_comb) ===
Range  : 9 – 142 mpg
Median : 22 mpg

Median MPG by Fuel Type:
Fuel Type
Electric              98.0
Hydrogen Fuel Cell    65.0
Hybrid                28.0
Natural Gas           28.0
Plug-in Hybrid        27.0
Diesel                25.0
Gas                   21.0
Flex-Fuel             17.0
Name: mpg_comb, dtype: float64


---

## Pillar 2: Performance

### Horsepower · Torque

In [168]:
print('=== HORSEPOWER ===')
print(f'Range  : {cars["hp"].min():.0f} – {cars["hp"].max():.0f} hp')
print(f'Median : {cars["hp"].median():.0f} hp')
print(f'Null   : {cars["hp"].isna().sum():,} ({cars["hp"].isna().mean():.1%}) — flagged, not imputed')
print()
print('=== TORQUE ===')
print(f'Range  : {cars["torque_lbft"].min():.0f} – {cars["torque_lbft"].max():.0f} lb-ft')
print(f'Median : {cars["torque_lbft"].median():.0f} lb-ft')
print(f'Null   : {cars["torque_lbft"].isna().sum():,} ({cars["torque_lbft"].isna().mean():.1%}) — flagged, not imputed')

=== HORSEPOWER ===
Range  : 27 – 1914 hp
Median : 266 hp
Null   : 3,181 (12.2%) — flagged, not imputed

=== TORQUE ===
Range  : 66 – 10296 lb-ft
Median : 262 lb-ft
Null   : 3,315 (12.7%) — flagged, not imputed


In [158]:
null_counts = cars.groupby('make').agg(
    hp_count=('hp', 'count'),
    hp_null=('hp', lambda x: x.isna().sum()),
    torque_count=('torque_lbft', 'count'),
    torque_null=('torque_lbft', lambda x: x.isna().sum()),
    trim=('trim', 'unique'),
    year=('year', 'unique')
)

null_counts[(null_counts['hp_count'] == 0) | (null_counts['torque_count'] == 0)]

,hp_count,hp_null,torque_count,torque_null,trim,year
make,,,,,,
Hummer,0,2,0,2,"[H3, H3T]",[2010]
Isuzu,0,4,0,4,[Hombre Pickup (FFV)],"[2001, 2002]"
Lordstown,1,0,0,1,[Endurance],[2023]
Lucid,15,0,0,15,"[Air Dream Edition Performance, Air Dream Edition Range, Air Grand Touring Performance, Air Grand Touring, Air Pure, Air Touring, Air Sapphire]","[2022, 2023, 2024]"
Mercury,0,44,0,44,"[Sable (FFV), Sable Wagon (FFV), Mountaineer FFV, Sable, Grand Marquis, Mariner Hybrid, Grand Marquis FFV, Mariner FFV, Milan FFV, Milan Hybrid]","[2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011]"
Plymouth,0,1,0,1,[Voyager/Grand Voyager],[2000]
Pontiac,0,4,0,4,[G6],"[2009, 2010]"
Saab,0,10,0,10,"[9-5, 9-3, 9-3 Sport, 9-3 SportCombi]","[2011, 2012]"
Saturn,0,10,0,10,"[Aura Hybrid, Relay, Vue Hybrid, Aura]","[2007, 2008, 2009, 2010]"


In [159]:
drop_idx = null_counts[(null_counts['hp_count'] == 0) | (null_counts['torque_count'] == 0)].index
cars = cars[~cars['make'].isin(drop_idx)]

---

## Pillar 3: Vehicle Type & Practicality

### Body Type · Drivetrain · Transmission · Cargo

In [160]:
print('=== BODY TYPE ===')
print(cars['bodytype'].value_counts(dropna=False).to_string())
print()
print('=== DRIVETRAIN ===')
print(cars['Drivetrain'].value_counts(dropna=False).to_string())
print()
print('=== TRANSMISSION TYPE ===')
print(cars['Transmission Type'].value_counts(dropna=False).to_string())
print()
print('=== DOORS ===')
print(cars['doors'].value_counts(dropna=False).sort_index().to_string())
print()
print('=== CARGO SPACE (non-Pickup) ===')
non_truck = cars.loc[cars['bodytype'] != 'Pickup', 'cargo_cuft']
print(f'Median : {non_truck.median():.1f} cu ft')
print(f'Range  : {non_truck.min():.1f} – {non_truck.max():.1f} cu ft')
print(f'Null   : {non_truck.isna().sum():,} ({non_truck.isna().mean():.1%}) — not imputed')

=== BODY TYPE ===
bodytype
SUV            7657
Sedan          6490
Pickup         3051
NaN            2837
Coupe          2009
Convertible    1381
Hatchback      1226
Wagon           911
Minivan         457
Van             134

=== DRIVETRAIN ===
Drivetrain
AWD    7573
FWD    7379
4WD    5041
RWD    4593
2WD    1567

=== TRANSMISSION TYPE ===
Transmission Type
Automatic    23849
Manual        1894
NaN            410

=== DOORS ===
doors
2.0     4324
3.0      154
4.0    21665
5.0       10

=== CARGO SPACE (non-Pickup) ===
Median : 19.1 cu ft
Range  : 3.3 – 313.9 cu ft
Null   : 4,851 (21.0%) — not imputed


---

## Pillar 4: Safety & Features

### Feature Availability & Luxury Gap

In [161]:
feat_filtered = features[features['car_id'].isin(cars['car_id'])].reset_index(drop=True)
feat_data_cols = [c for c in feat_filtered.columns if c not in ('car_id', 'review_id')]

# NaN = feature not listed → treat as Not Available (0)
feat_filtered[feat_data_cols] = feat_filtered[feat_data_cols].fillna(0).astype(int)

availability = (
    feat_filtered[feat_data_cols]
    .apply(lambda col: pd.Series({
        'standard_%':  (col == 2).mean() * 100,
        'optional_%':  (col == 1).mean() * 100,
        'not_avail_%': (col == 0).mean() * 100,
    }))
    .T
    .round(1)
    .sort_values('standard_%', ascending=False)
)
availability

,standard_%,optional_%,not_avail_%
Child Seat Anchors,82.2,0.2,17.6
Power Windows,82.2,3.7,14.1
Tilt Steering Wheel,81.3,1.7,17.0
Cruise Control,77.3,5.6,17.1
Alloy Wheels,74.7,9.3,16.0
Stability Control,74.1,1.1,24.8
Traction Control,72.5,1.9,25.5
Cup Holder Count,68.1,0.2,31.6
Remote Keyless Entry,68.1,3.7,28.2
Child Door Locks,67.4,0.0,32.6


In [162]:
# Luxury vs Non-Luxury feature gap
merged_feat = feat_filtered.merge(cars[['car_id', 'luxury']], on='car_id')
feat_by_luxury = (
    merged_feat.groupby('luxury')[feat_data_cols]
    .apply(lambda df: (df == 2).mean() * 100)
    .T
    .round(1)
)
feat_by_luxury.columns = ['non_luxury_std%', 'luxury_std%']
feat_by_luxury['luxury_advantage'] = (feat_by_luxury['luxury_std%'] - feat_by_luxury['non_luxury_std%']).round(1)
feat_by_luxury.sort_values('luxury_advantage', ascending=False)

,non_luxury_std%,luxury_std%,luxury_advantage
Dual Power Front Seats,24.0,77.5,53.5
Rain Sensing Windshield Wipers,12.3,63.8,51.5
Leather Seats,29.9,65.7,35.8
Power Driver's Seat,45.2,80.8,35.6
Power Folding Exterior Mirrors,8.8,43.7,34.9
Navigation System,19.6,50.6,31.0
Premium Radio,16.8,44.9,28.1
Tilt/Telescoping Steering Wheel,55.5,77.6,22.1
Alarm System,48.7,68.6,19.9
Heated Mirrors,50.6,70.5,19.9


---

## Pillar 5: Brand & Quality

### EV Adoption by Year

In [163]:
cars_full = pd.read_csv(BASE / 'data/final/cars/car_stats.UVCRS')
cars_full['is_ev'] = cars_full['Fuel Type'].str.contains('Electric', case=False, na=False)

ev_by_year = (
    cars_full.groupby('year')['is_ev']
    .agg(ev_trims='sum', total='count')
)
ev_by_year['ev_pct'] = (ev_by_year['ev_trims'] / ev_by_year['total'] * 100).round(1)
ev_by_year[ev_by_year['total'] >= 5]

,ev_trims,total,ev_pct
year,,,
2000,4,25,16.0
2001,5,386,1.3
2002,2,422,0.5
2003,1,582,0.2
2004,0,676,0.0
2005,0,742,0.0
2006,0,759,0.0
2007,0,791,0.0
2008,1,856,0.1


In [164]:
print('=== BRAND DISTRIBUTION (top 15 by trim count) ===')
make_counts = cars['make'].value_counts()
print(f'Total makes: {make_counts.shape[0]}')
print()
print(make_counts.head(15).to_string())
print()
lux_makes = cars[cars['luxury'] == 1]['make'].nunique()
print(f'Luxury makes: {lux_makes} of {make_counts.shape[0]} total')

=== BRAND DISTRIBUTION (top 15 by trim count) ===
Total makes: 54

make
Ford             2164
Chevrolet        1999
Toyota           1750
Nissan           1497
BMW              1474
Audi             1268
GMC              1219
Volkswagen       1034
Mercedes-Benz     963
Porsche           938
Honda             936
Jeep              931
Dodge             806
Subaru            804
Kia               746

Luxury makes: 23 of 54 total


---

## Consumer View

Focused dataset joining consumer-relevant columns with review quality ratings.

In [165]:
CONSUMER_COLS_FINAL = [
    'car_id', 'make', 'model', 'year', 'trim', 'bodytype', 'doors',
    'price', 'Fuel Type', 'Drivetrain', 'Transmission Type',
    'mpg_comb', 'hp', 'hp_missing', 'torque_lbft', 'torque_missing',
    'cargo_cuft', 'is_ev', 'luxury',
]

consumer_view = cars[CONSUMER_COLS_FINAL].copy()

# Join review ratings
if 'review_id' in cars.columns:
    consumer_view = consumer_view.merge(
        cars[['car_id', 'review_id']].drop_duplicates()
            .merge(reviews, on='review_id', how='left')
            .drop(columns='review_id'),
        on='car_id', how='left'
    )

print(f'Consumer view shape: {consumer_view.shape}')
consumer_view.head(3)

Consumer view shape: (26153, 25)


,car_id,make,model,year,trim,bodytype,doors,price,Fuel Type,Drivetrain,Transmission Type,mpg_comb,hp,hp_missing,torque_lbft,torque_missing,cargo_cuft,is_ev,luxury,rating_value,rating_performance,rating_quality,rating_comfort,rating_reliability,rating_styling
0,CHR00TOW01,Chrysler,Town and Country/Voyager/Grand Voy.,2000,Town and Country/Voyager/Grand Voy.,NaN,4.0,6675.0,Flex-Fuel,FWD,Automatic,18.0,NaN,1,NaN,1,NaN,False,0,NaN,NaN,NaN,NaN,NaN,NaN
1,DOD00CAR01,Dodge,Caravan/Grand Caravan,2000,Caravan/Grand Caravan,NaN,4.0,10500.0,Flex-Fuel,FWD,Automatic,19.0,NaN,1,NaN,1,NaN,False,0,NaN,NaN,NaN,NaN,NaN,NaN
2,FOR00EXP01,Ford,Explorer,2000,Explorer,NaN,4.0,10950.0,Flex-Fuel,RWD,Automatic,15.0,NaN,1,NaN,1,NaN,False,0,NaN,NaN,NaN,NaN,NaN,NaN


In [166]:
print('=== Consumer View — Null Summary ===')
null_check = pd.DataFrame({
    'non_null': consumer_view.notna().sum(),
    'null':     consumer_view.isna().sum(),
    'null_%':   (consumer_view.isna().mean() * 100).round(1),
})
null_check[null_check['null'] > 0].sort_values('null_%', ascending=False)

=== Consumer View — Null Summary ===


,non_null,null,null_%
cargo_cuft,18258,7895,30.2
torque_lbft,22838,3315,12.7
hp,22972,3181,12.2
bodytype,23316,2837,10.8
rating_value,23699,2454,9.4
rating_performance,23699,2454,9.4
rating_quality,23699,2454,9.4
rating_comfort,23699,2454,9.4
rating_reliability,23699,2454,9.4
rating_styling,23699,2454,9.4


In [172]:
cars.isna().sum()

review_id               1737
car_id                     0
make                       0
model                      0
year                       0
bodytype                2837
doors                      0
trim                       0
price                     93
mpg_city                 711
mpg_hwy                  710
mpg_comb                 439
Fuel Type                  0
hp                      3181
Engine                   996
torque_lbft             3315
Recommended Fuel        4680
Transmission Type        410
Drivetrain                 0
0 - 60                 19085
curb_weight_lbs         4456
cargo_cuft              7895
Fuel Capacity           3424
Wheel Base              2990
Overall Length          3072
Front Head Room         4527
Front Leg Room          4840
Front Shoulder Room     4834
Width with mirrors      7064
Turning Diameter        5821
is_ev                      0
luxury                     0
hp_missing                 0
torque_missing             0
dtype: int64

In [167]:
OUT = BASE / 'data/final/consumer_view.UVCRS'
consumer_view.to_csv(OUT, index=False)
print(f'Saved  → {OUT}')
print(f'Shape  : {consumer_view.shape}')

Saved  → /Users/eric5805/Downloads/csv/data/final/consumer_view.csv
Shape  : (26153, 25)
